In [ ]:

%pip install -qU "transformers[torch]" datasets[audio] accelerate evaluate sentencepiece jiwer torch torchaudio torchcodec peft



In [ ]:
%pip freeze >> requirements.txt

In [ ]:
import transformers, datasets, torch, torchaudio, evaluate, jiwer, torchcodec

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("torchcodec:", torchcodec.__version__)



transformers: 4.57.3
torch: 2.9.1+cu128
torchcodec: 0.9.0


In [ ]:


from datasets import Audio
print("torch:", torch.__version__)
print("torchcodec:", torchcodec.__version__)
from google.colab import drive
import zipfile
import shutil
import torchaudio
import datasets
from datasets import Dataset
import pandas as pd
drive.mount('/content/drive', force_remount=True)
import os



torch: 2.9.1+cu128
torchcodec: 0.9.0
Mounted at /content/drive


In [ ]:
root="/content/drive/MyDrive/Caribbean_ASR_Hackathon/data/"
def imp_cast(pathend):

  csv_path=root+pathend
  df=pd.read_csv(csv_path)
  df['audio_path']=root+"audio_files/" + df["ID"] + ".wav"
  df=df.rename(columns={"Transcription": "text"})
  # Filter out missing files
  df=df[df['audio_path'].apply(os.path.exists)]
  print(len(df))
  ds = Dataset.from_pandas(df[["audio_path", "text"]])
  ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
  ds = ds.rename_column("audio_path", "audio")
  return ds



In [ ]:

train_ds=imp_cast("Train.csv")
#eval_ds=imp_cast("Test.csv") #use full training set for final submission

ds = datasets.DatasetDict({"train": train_ds})


19855


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-large-v3"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="en", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.task = "transcribe"
model.generation_config.language = "en"       # accents, but still English

model.config.forced_decoder_ids = None        # allow free decoding
model.config.suppress_tokens = []             # don’t suppress anything extra


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:

#Prep Training set
def prepare_batch(batch):
    # audio
    audio = batch["audio"]
    inputs = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    )

    # text → labels (no as_target_processor)
    labels = processor.tokenizer(
        batch["text"],
        return_attention_mask=False,
    ).input_ids

    batch["input_features"] = inputs["input_features"][0]
    batch["labels"] = labels
    return batch

encoded = ds.map(
    prepare_batch,

)

#encoded.save_to_disk(root+'LORA32/encoded_split_CS')
#from datasets import load_dataset
#encoded=load_dataset(root+'LORA32/encoded_split_CS')

Map:   0%|          | 0/19855 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data collator that:
    - Pads audio features to same length
    - Pads label sequences to same length
    - Replaces label padding with -100 so it’s ignored in the loss
    """
    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        # 1) Audio features (already computed as "input_features")
        model_input_name = self.processor.model_input_names[0]  # usually "input_features"
        input_features = [
            {model_input_name: f[model_input_name]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",return_attention_mask=True
        )

        # 2) Text labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        # 3) Replace padding token with -100
        labels = labels_batch["input_ids"]
        attention_mask = labels_batch["attention_mask"]
        labels = labels.masked_fill(attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # attention weights
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
import evaluate
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback
import os
os.environ["WANDB_DISABLED"] = "true"
wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    preds = pred_str[:5]
    refs = label_str[:5]
    for p, r in zip(preds, refs):
        print("PRED:", p)
        print("REF :", r)
        print("----")

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


# -----------------------------------------------------------

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-caribbean",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    warmup_steps=500,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    max_steps=-1,
    logging_steps=50,


)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    compute_metrics=compute_metrics,


)

trainer.train()
trainer.save_model(root+'LORA32/model_full')
processor.save_pretrained(root+'LORA32/model_full')


In [ ]:
#compute wer
'''
eval_preds=trainer.predict(encoded['test'])
pred_ids = eval_preds.predictions
# Decode to text
pred_texts = processor.batch_decode(
    pred_ids,
    skip_special_tokens=True
)

# Example: print first 5 transcripts
for i, txt in enumerate(pred_texts[:5]):
    print(f"[{i}] {txt}")
from evaluate import load
wer_metric = load("wer")

label_ids = eval_preds.label_ids
label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

ref_texts = processor.batch_decode(label_ids, skip_special_tokens=True)
wer = wer_metric.compute(predictions=pred_texts, references=ref_texts)
print("Test WER:", wer)
'''


In [ ]:
#import test dataset and predict on it to create submission

# ==========================================
# 3. Load Test.csv
# ==========================================
import pandas as pd

test_csv_path = '/content/drive/MyDrive/CARIB_VOICE/Test.csv'
test_df = pd.read_csv(test_csv_path)

print("Test DataFrame head:")
print(test_df.head())
print("Total test files:", len(test_df))

In [ ]:

import time
import librosa


AUDIO_SR = 16000
MAX_SECONDS = 30
MAX_LEN = int(MAX_SECONDS * AUDIO_SR)
audio_dir = "/content/drive/MyDrive/Caribbean_ASR_Hackathon/data/audio_files"
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Enable some perf knobs
torch.backends.cuda.matmul.allow_tf32 = True  # A100 supports TF32
torch.backends.cudnn.benchmark = True

# Force Whisper to transcribe in English
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="en",      # change if needed
    task="transcribe",
)

BATCH_SIZE = 8   # try 8/16, tune to your VRAM
predictions = []

ids = test_df["ID"].tolist()
paths = [os.path.join(audio_dir, f"{audio_id}.wav") for audio_id in ids]

start_time = time.time()
n_files = len(paths)

with torch.inference_mode():  # faster than no_grad for inference
    for start in range(0, n_files, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_files)
        batch_paths = paths[start:end]

        batch_audio = []
        batch_valid_idx = []  # which indices are valid (file exists)

        for i, p in enumerate(batch_paths):
            global_idx = start + i

            if not os.path.exists(p):
                print(f"Warning: missing audio file: {p}")
                batch_audio.append(None)
                continue

            if global_idx % 100 == 0:
                elapsed = time.time() - start_time
                print(f"Processing file {global_idx}/{n_files}  |  Elapsed: {elapsed:.1f}s")

            # Load and truncate
            speech_array, sr = librosa.load(p, sr=AUDIO_SR)
            if len(speech_array) > MAX_LEN:
                speech_array = speech_array[:MAX_LEN]

            batch_audio.append(speech_array)
            batch_valid_idx.append(global_idx)
        '''
        # If all files in this batch were missing, just extend with empties
        if all(a is None for a in batch_audio):
            predictions.extend([""] * (end - start))
            continue
        '''

        # Filter out missing ones for processing
        audio_for_proc = [a for a in batch_audio if a is not None
        ]

        inputs = processor(
            audio_for_proc,
            sampling_rate=AUDIO_SR,
            return_tensors="pt",
            padding=True,
        )
        input_features = inputs.input_features.to(device)

        # Generation (greedy, short max_new_tokens for speed)
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=256,   # shrink if your clips are short
            num_beams=1,          # greedy
            do_sample=False,
        )

        texts = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True,
        )

        # Now write predictions back in order, including "" for missing files
        batch_preds = [""] * (end - start)
        j = 0
        for i, a in enumerate(batch_audio):
            if a is not None:
                batch_preds[i] = texts[j].strip()
                j += 1
            else:
                batch_preds[i] = ""

        predictions.extend(batch_preds)

total_time = time.time() - start_time
print("Number of predictions:", len(predictions))
print(f"Total inference time: {total_time/60:.1f} minutes")

print("\nSample predictions:")
for i in range(min(5, len(predictions))):
    print(f"ID: {test_df['ID'].iloc[i]}, Pred: '{predictions[i]}'")


In [ ]:
submission_df = pd.DataFrame({
    "ID": test_df["ID"],
    "Transcription": predictions
})

submission_path = "/content/drive/MyDrive/Caribbean_ASR_Hackathon/data/LORA32/submission_full.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Saved submission file to: {submission_path}")
print(submission_df.head())

In [ ]:
'''
double check prepare batch worked
submit with this lora
write python script to identify errors, then fine tune only on those
normalize all to lower case
copy this notebook and do data aug/ more epochs w/errors oversampled


'''



In [ ]:

import gc, torch
gc.collect()
torch.cuda.empty_cache()
# delete big objects
#del model
#del trainer
# del anything else big you know you moved to cuda

